# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [3]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

# MY SOLUTION:

## Step 1:
- Create two tables from the original patent table for easy filtering. \
      * Table 1 (cited) = Patent # and POSTATE of the CITED patents \
      * Table 2 (citing) = Patent # and the POSTATE of the CITING patent
- Join the two tables together using the CITATIONS table that identifies which patent's have CITED another patent. \
      * Produce the joined table that connects the CITING and CITED patents

In [72]:
# Create 2 tables that only hold the patent# and the state, 1 for the CITED # and 1 for the CITING #
cited = patents.select("PATENT", col("POSTATE").alias("CITED_STATE"))
citing = patents.select("PATENT",col("POSTATE").alias("CITING_STATE"))

# Join the tables together matching the CITING# to the CITING PATENT#, and the CITED# to the CITED PATENT #
joined = citations.join(citing, citations.CITING == citing.PATENT).join(cited, citations.CITED == cited.PATENT)


In [35]:
joined.show(10)

+-------+-------+-------+------------+-------+-----------+
| CITING|  CITED| PATENT|CITING_STATE| PATENT|CITED_STATE|
+-------+-------+-------+------------+-------+-----------+
|4483021|3070803|4483021|          MS|3070803|         IL|
|4133055|3070803|4133055|          NH|3070803|         IL|
|4253313|3070803|4253313|        NULL|3070803|         IL|
|5054122|3070803|5054122|        NULL|3070803|         IL|
|5557807|3070803|5557807|          FL|3070803|         IL|
|4484363|3070803|4484363|          CA|3070803|         IL|
|4921141|3070803|4921141|          CA|3070803|         IL|
|5469579|3070803|5469579|        NULL|3070803|         IL|
|5850636|3070803|5850636|          CA|3070803|         IL|
|4400830|3070805|4400830|          FL|3070805|         CA|
+-------+-------+-------+------------+-------+-----------+
only showing top 10 rows



## Step 2:
- Clean up the joined table to display a clean non-null containing table in the form of: \
      CITING | CITING_STATE | CITED | CITED_STATE \
  With matching CITING_STATE and CITED_STATE columns

In [73]:
# Remove the extra columns and produce a table that is only CITING | CITING_STATE | CITED | CITED_STATE
intermediateTable = joined.select("CITING", "CITING_STATE", "CITED", "CITED_STATE")

In [46]:
intermediateTable.show(10)

+-------+------------+-------+-----------+
| CITING|CITING_STATE|  CITED|CITED_STATE|
+-------+------------+-------+-----------+
|4483021|          MS|3070803|         IL|
|4133055|          NH|3070803|         IL|
|4253313|        NULL|3070803|         IL|
|5054122|        NULL|3070803|         IL|
|5557807|          FL|3070803|         IL|
|4484363|          CA|3070803|         IL|
|4921141|          CA|3070803|         IL|
|5469579|        NULL|3070803|         IL|
|5850636|          CA|3070803|         IL|
|4400830|          FL|3070805|         CA|
+-------+------------+-------+-----------+
only showing top 10 rows



In [74]:
filteredTable = intermediateTable.filter(col("CITING_STATE")==col("CITED_STATE")).filter(col("CITING_STATE").isNotNull()).filter(col("CITED_STATE").isNotNull())

In [67]:
filteredTable.show(10)

+-------+------------+-------+-----------+
| CITING|CITING_STATE|  CITED|CITED_STATE|
+-------+------------+-------+-----------+
|4067198|          AK|3217791|         AK|
|4676695|          AK|3217791|         AK|
|5190098|          AK|3217791|         AK|
|5238053|          AK|3217791|         AK|
|4075779|          AK|3373523|         AK|
|4130086|          AK|3464385|         AK|
|4178878|          AK|3464385|         AK|
|4344414|          AK|3472314|         AK|
|5618134|          AK|3472314|         AK|
|4205718|          AK|3472314|         AK|
+-------+------------+-------+-----------+
only showing top 10 rows



## Step 3:
- Take the clean table and produce a table that groups each entry by the CITING patent number, count the entries pertaining to each of the CITING patents that will be passed to the final table display

In [75]:
countedCiting = filteredTable.groupBy("CITING").count().withColumnRenamed("count", "SAME_STATE")

## Step 4:
- Join the counted table with the original patent table, remove extra column of CITING number, and display the results in descending order based off the count (SAME_STATE) column from the counted table.

In [76]:
finalTable = patents.join(countedCiting, patents.PATENT == countedCiting.CITING)

In [77]:
finalTable.sort("SAME_STATE", ascending=False).drop("CITING").show(10)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|SAME_STATE|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|5959466| 1999|14515|   1997|     US|     CA|    5310|      2|  NULL|   326|  4|    46|  159|       0|     1.0|   NULL|  0.6186|    NULL|  4.8868|  0.0455|   0.044|    NULL|    NULL|       125|
|5983822| 1999|14564|   1998|     US|     TX|  569900|      2|  NULL|   114|  5|    55|  200|       0|   0.995|   NULL|  0.7201|    NULL|   12.45|     0.0|     0.0|    NULL|    NULL|       103|
|6008204| 1999|14606|   1998| 